In [1]:
import os, sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("utils"), "..")))
from module.utils import *
from module.prompt import *
from module.custom_model import *
from module.base_model import *
from module.tools import *

start_langsmith("final_music")
from typing import TypedDict, Annotated, List, Literal, Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks

LangSmith 추적을 시작합니다.
[프로젝트명]
final_music


In [2]:
class State(TypedDict):
    question: Annotated[str, "user input question"]  # 사용자 질의 or requeustion 질의
    plan: Annotated[list[str], "get plan_node"]  # llm 생성한 작업 계획서
    messages: Annotated[list, add_messages]  # 작업 수행 후 얻은 데이터
    past_steps: Annotated[list, add_messages]  # 현재 단계에서 수행한 결과
    answer: Annotated[str, " output final answer"]  # 최종 답변 출력
    human_feedback: bool = False
    next_agent: Annotated[str, " use agent"]

In [3]:
path_template = """
    # System :
    You are an AI assistant 
    You can judge the user's input and Ai answer, make decisions

    # Previous AI Answers :
    {answer}

    # User input :
    {question}

    
    # Rules :
    Here are two parameters
    The question parameter includes a user's question or answer

    In the case of answer, include the answer

    Understand the meaning of question and answer, answer request_node if the user requests, and answer response_node if the user is responding to an AI request


    # Important :
    The answer must be one of three
    The answer is available in Korean
"""
path_prompt = ChatPromptTemplate.from_template(path_template)

In [4]:
def path_node(state: State) -> State:
    llm = get_gpt()
    prompt = path_prompt
    chain = prompt | llm.with_structured_output(PathQuery)
    response = chain.invoke(
        {"question": [state["question"]], "answer": [state["answer"]]}
    )
    return State({"messages": response.datasource})

In [5]:
state_graph = StateGraph(State)
state_graph.add_node("path_node", path_node)

state_graph.add_edge(START, "path_node")
state_graph.add_edge("path_node", END)

cp = get_check_pointer()
graph = state_graph.compile(checkpointer=cp)

In [6]:
# visualize_graph(graph)

In [17]:
config = get_runnable_config(recursion_limit=20, thread_id=get_random_uuid())
query = " 아니 db_agent사용해"
answer = ""
answer = " sub_graph를 사용하여 favorite music 을 실행할까요?"
inputs = {"question": HumanMessage(content=query), "answer": AIMessage(content=answer)}
response = graph.invoke(input=inputs, config=config, stream_mode="values")
print(response)

{'question': HumanMessage(content=' 아니 db_agent사용해', additional_kwargs={}, response_metadata={}), 'messages': [HumanMessage(content='response_node', additional_kwargs={}, response_metadata={}, id='37bec5c9-9527-47d5-92f1-5f612c3c4c52')], 'past_steps': [], 'answer': AIMessage(content=' sub_graph를 사용하여 favorite music 을 실행할까요?', additional_kwargs={}, response_metadata={})}
